# BNPL Governance Workshop - Confluent Cloud Schema Registry

This hands-on notebook demonstrates advanced capabilities of Confluent Cloud Schema Registry for Buy-Now-Pay-Later (BNPL) governance, including:
1. Schema Creation & Evolution
2. Producing data with invalid and valid schemas
3. Enforcing Data Contracts (Rules)
4. Producing data with valid and invalid data contracts
5. Client-Side Field Level Encryption (CSFLE) in Confluent Cloud
6. Consuming with and without CSFLE decryption

**Note:** Ensure you have your `.env` file saved in the same directory as this notebook.

### Install dependencies

Installs the Kafka client, `.env` loader, and JSON Schema validator libraries used throughout this notebook.

**Example:** after this runs, `import confluent_kafka` and `from jsonschema.exceptions import ValidationError` succeed in later cells.

### Connect to Confluent Cloud

Loads credentials from `.env`, builds the Kafka client config and the Schema Registry client, then creates a fresh topic to use for the rest of the notebook (auto topic creation is disabled on Confluent Cloud, so this step is required).

**Example:** `topic_name` ends up something like `bnpl_transactions_ahartono` -- namespaced by your `PARTICIPANT_ID` (or OS username) so multiple people running this notebook don't collide on the same topic, and so the cleanup cell at the end can always reconstruct the exact name later.

In [ ]:
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy PIP_CMD and run it yourself in a terminal (with .venv activated)

PIP_CMD = "pip install -q confluent-kafka python-dotenv jsonschema"

if RUN_IN_NOTEBOOK:
    !{PIP_CMD}
else:
    print(f"Skipping -- run this yourself in a terminal (with .venv activated):\n\n  $ {PIP_CMD}")

In [2]:
import os
import uuid
from dotenv import load_dotenv
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka.schema_registry import SchemaRegistryClient, Schema

# Load Confluent Cloud credentials from .env
load_dotenv(".env")

bootstrap_servers = os.getenv("CCLOUD_BOOTSTRAP_SERVERS")
kafka_api_key = os.getenv("CCLOUD_API_KEY")
kafka_api_secret = os.getenv("CCLOUD_API_SECRET")
sr_url = os.getenv("SCHEMA_REGISTRY_URL")
sr_api_key = os.getenv("SCHEMA_REGISTRY_API_KEY")
sr_api_secret = os.getenv("SCHEMA_REGISTRY_API_SECRET")

# Kafka client configuration
kafka_conf = {
    'bootstrap.servers': bootstrap_servers,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': kafka_api_key,
    'sasl.password': kafka_api_secret
}

# Schema Registry configuration
sr_conf = {
    'url': sr_url,
    'basic.auth.user.info': f"{sr_api_key}:{sr_api_secret}"
}

sr_client = SchemaRegistryClient(sr_conf)

# Namespaced by PARTICIPANT (same convention as kredivo-02/03) so topic names are
# deterministic -- no random suffix to track down when cleaning up later.
PARTICIPANT = os.getenv("PARTICIPANT_ID") or os.getenv("USER") or "p00"
PARTICIPANT = "".join(c for c in PARTICIPANT if c.isalnum() or c in "-_")[:20] or "p00"

topic_name = f"bnpl_transactions_{PARTICIPANT}"

def create_topic(name, num_partitions=1):
    """Confluent Cloud clusters typically have auto topic creation disabled,
    so producing to a not-yet-existing topic fails with UNKNOWN_TOPIC_OR_PART."""
    admin = AdminClient(kafka_conf)
    fs = admin.create_topics([NewTopic(name, num_partitions=num_partitions, replication_factor=3)])
    for topic, f in fs.items():
        try:
            f.result()
            print(f"Created topic: {topic}")
        except Exception as e:
            print(f"Topic creation note for {topic}: {e}")

create_topic(topic_name)

print(f"Loaded configurations from .env successfully!")
print(f"Participant namespace: {PARTICIPANT}")
print(f"Target Workshop Topic: {topic_name}")

Created topic: bnpl_transactions_c52661
Loaded configurations from .env successfully!
Target Workshop Topic: bnpl_transactions_c52661


## 1. Schema Creation & Evolution
First, we will create a base JSON schema (V1) for our transactions and register it.
Next, we will evolve the schema to V2 by adding a new `merchant` field.

### Set up the JSON serializer

Creates a `JSONSerializer` bound to schema V2 and a `Producer`. Every message passed through this serializer is validated against the schema *before* it's allowed onto the wire.

**Example:** `json_serializer_v2({"transaction_id": "1", "amount": 10, "status": "OK"}, ctx)` returns schema-validated bytes ready for `producer.produce()`; a dict missing `amount` raises `ValidationError` instead.

### Try sending invalid data

`invalid_data` is missing the required `amount` field. Schema validation rejects it before it ever reaches Kafka, proving the schema is actively enforced — not just documentation.

**Example:** expect `Error: 'amount' is a required property` in the output; no message is delivered.

### Send valid data

Same flow as above, but `valid_data` includes every required field plus the optional `merchant`, so it passes validation and is delivered to the topic.

**Example:** output shows `Message delivered to bnpl_transactions_... [0] @ offset 0`.

In [3]:
# V1 Schema Creation
schema_v1_str = """{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Transaction",
  "type": "object",
  "properties": {
    "transaction_id": {"type": "string"},
    "amount": {"type": "number"},
    "status": {"type": "string"}
  },
  "required": ["transaction_id", "amount"],
    "additionalProperties": false
}"""

schema_v1 = Schema(schema_v1_str, schema_type="JSON")
subject_name = f"{topic_name}-value"

schema_id_v1 = sr_client.register_schema(subject_name, schema_v1)
print(f"Registered Schema V1 with ID: {schema_id_v1}")

# V2 Schema Evolution
schema_v2_str = """{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Transaction",
  "type": "object",
  "properties": {
    "transaction_id": {"type": "string"},
    "amount": {"type": "number"},
    "status": {"type": "string"},
    "merchant": {"type": "string"}
  },
  "required": ["transaction_id", "amount"],
    "additionalProperties": false
}"""

schema_v2 = Schema(schema_v2_str, schema_type="JSON")
schema_id_v2 = sr_client.register_schema(subject_name, schema_v2)
print(f"Registered Schema V2 with ID: {schema_id_v2}")

Registered Schema V1 with ID: 100006
Registered Schema V2 with ID: 100007


## 2 & 3. Produce Data using Invalid and Valid Data
Let's initialize the Producer and JSONSerializer, then attempt to send valid and invalid data against our standard schema.

In [4]:
from confluent_kafka.schema_registry.json_schema import JSONSerializer
from confluent_kafka.serialization import SerializationContext, MessageField
from jsonschema.exceptions import ValidationError

json_serializer_v2 = JSONSerializer(schema_v2_str, sr_client)
producer = Producer(kafka_conf)

def delivery_report(err, msg):
    if err is not None:
        print(f"Delivery failed: {err}")
    else:
        print(f"Message delivered to {msg.topic()} [{msg.partition()}] @ offset {msg.offset()}")

In [5]:
# Produce Invalid Data (Missing 'amount')
invalid_data = {
    "transaction_id": str(uuid.uuid4()),
    "status": "PENDING"
}

try:
    print("Attempting to produce invalid data...")
    ctx = SerializationContext(topic_name, MessageField.VALUE)
    serialized_invalid = json_serializer_v2(invalid_data, ctx)
    producer.produce(topic=topic_name, value=serialized_invalid, on_delivery=delivery_report)
    producer.flush()
except ValidationError as e:
    print(f"\n[Schema Validation Error as Expected]: {e.message}")
except Exception as e:
    print(f"Error: {e}")

Attempting to produce invalid data...
Error: 'amount' is a required property


In [6]:
# Produce Valid Data
valid_data = {
    "transaction_id": str(uuid.uuid4()),
    "amount": 150.75,
    "status": "APPROVED",
    "merchant": "TechStore"
}

try:
    print("\nAttempting to produce valid data...")
    ctx = SerializationContext(topic_name, MessageField.VALUE)
    serialized_valid = json_serializer_v2(valid_data, ctx)
    producer.produce(topic=topic_name, value=serialized_valid, on_delivery=delivery_report)
    producer.flush()
    print("Successfully produced valid data!")
except Exception as e:
    print(f"Unexpected Error: {e}")


Attempting to produce valid data...
Message delivered to bnpl_transactions_c52661 [0] @ offset 0
Successfully produced valid data!


%6|1786457944.927|GETSUBSCRIPTIONS|rdkafka#producer-2| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to fKYcq0YAQ1a1VEQQO02WzA


## 4. Show the Data Contract
Data Contracts extend schemas with rules (e.g., Google Common Expression Language - CEL). We'll add a contract ruling that `amount` must be greater than `0` and less than `10000`.

In [7]:
from confluent_kafka.schema_registry import Rule, RuleSet, RuleKind, RuleMode, RuleParams

amount_rule = Rule(
    name="validate_amount",
    doc="Transaction amount must be between 0 and 10000",
    kind=RuleKind.CONDITION,
    mode=RuleMode.WRITE,
    type="CEL",
    tags=None,
    params=None,
    expr="message.amount > 0 && message.amount < 10000",
    on_success=None,
    on_failure="ERROR",
    disabled=False,
)

rule_set = RuleSet(migration_rules=None, domain_rules=[amount_rule])

schema_with_contract = Schema(
    schema_str=schema_v2_str, 
    schema_type="JSON", 
    rule_set=rule_set
)

contract_schema_id = sr_client.register_schema(subject_name, schema_with_contract)
print(f"Registered Schema with Data Contract (CEL Rule) ID: {contract_schema_id}")

# Ensure our serializer pulls the latest contract
json_serializer_contract = JSONSerializer(
    schema_str=schema_v2_str, 
    schema_registry_client=sr_client,
    conf={'auto.register.schemas': False, 'use.latest.version': True}
)


Registered Schema with Data Contract (CEL Rule) ID: 100008


## 5 & 6. Produce with Valid and Invalid Data Contract
We will test our `amount > 0 && amount < 10000` contract.

In [8]:
# Valid Data Contract Produce
valid_contract_data = {
    "transaction_id": str(uuid.uuid4()),
    "amount": 250.00,  # Valid amount
    "merchant": "ValidStore"
}

try:
    # Register the CEL rule executor so the engine can evaluate the CONDITION rule.
    from confluent_kafka.schema_registry.rules.cel.cel_executor import CelExecutor
    CelExecutor.register()

    ctx = SerializationContext(topic_name, MessageField.VALUE)
    serialized = json_serializer_contract(valid_contract_data, ctx)
    producer.produce(topic=topic_name, value=serialized, on_delivery=delivery_report)
    producer.flush()
    print("Successfully produced record obeying Data Contract!")
except Exception as e:
    print(f"Error ({type(e).__name__}): {e}")


Message delivered to bnpl_transactions_c52661 [0] @ offset 1
Successfully produced record obeying Data Contract!


### Try violating the contract

Same producer/serializer as before, but `amount` is `-50.00`. That's still valid JSON (a `number`), so plain schema validation would let it through — but it fails the CEL rule `amount > 0`, which is exactly what data contracts add on top of schemas.

**Example:** expect `[Data Contract Validation Error as Expected]` instead of a delivered offset.

In [9]:
# Invalid Data Contract Produce
invalid_contract_data = {
    "transaction_id": str(uuid.uuid4()),
    "amount": -50.00,  # Invalid: violates amount > 0
    "merchant": "InvalidStore"
}

try:
    ctx = SerializationContext(topic_name, MessageField.VALUE)
    serialized = json_serializer_contract(invalid_contract_data, ctx)
    producer.produce(topic=topic_name, value=serialized, on_delivery=delivery_report)
    producer.flush()
except Exception as e:
    print(f"\n[Data Contract Validation Error as Expected]: {e}")


[Data Contract Validation Error as Expected]: 


## 7. Show how to use CSFLE in Confluent Cloud
Client-Side Field Level Encryption (CSFLE) lets you encrypt sensitive fields automatically. We'll use a `LocalKmsDriver` and tag the `national_id` field as `PII` to trigger encryption.

In [10]:
schema_csfle_str = """{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Customer",
  "description": "BNPL customer record with PII field-level encryption",
  "type": "object",
  "properties": {
    "customer_id": {"type": "string"},
    "national_id": {
        "type": "string",
        "confluent:tags": ["PII"]
    }
  },
  "required": ["customer_id", "national_id"],
  "additionalProperties": false
}"""

# Stream Governance rejects a schema whose embedded confluent:tags aren't already
# defined in the environment's Data Catalog. Create the PII tag def if missing.
def ensure_tag_defs(names, entity_types=("sr_schema", "sr_record", "sr_field")):
    existing = requests.get(f"{sr_url}/catalog/v1/types/tagdefs", auth=(sr_api_key, sr_api_secret), timeout=30)
    existing.raise_for_status()
    have = {t["name"] for t in existing.json()}
    missing = [n for n in names if n not in have]
    if not missing:
        return
    body = [{"entityTypes": list(entity_types), "name": n} for n in missing]
    resp = requests.post(f"{sr_url}/catalog/v1/types/tagdefs", auth=(sr_api_key, sr_api_secret),
                          headers={"Content-Type": "application/json"}, json=body, timeout=30)
    resp.raise_for_status()

import requests
ensure_tag_defs(["PII"])

encrypt_rule = Rule(
    name="encrypt_pii",
    doc="Encrypt fields tagged with PII",
    kind=RuleKind.TRANSFORM,
    mode=RuleMode.WRITEREAD,
    type="ENCRYPT",
    tags=["PII"],
    params=RuleParams({
        "encrypt.kms.type": "local-kms",
        "encrypt.kms.key.id": "my-local-key",
        "encrypt.kek.name": "my-local-kek",
    }),
    expr=None,
    on_success=None,
    on_failure="ERROR,NONE",
    disabled=False,
)

csfle_rule_set = RuleSet(migration_rules=None, domain_rules=[encrypt_rule])
schema_csfle = Schema(schema_str=schema_csfle_str, schema_type="JSON", rule_set=csfle_rule_set)

topic_csfle = f"{topic_name}_secure"
csfle_subject_name = f"{topic_csfle}-value"
create_topic(topic_csfle)

csfle_schema_id = sr_client.register_schema(csfle_subject_name, schema_csfle)
print(f"Registered CSFLE Schema ID: {csfle_schema_id}")

# Setup Local KMS Driver
from confluent_kafka.schema_registry.rules.encryption.encrypt_executor import FieldEncryptionExecutor
from confluent_kafka.schema_registry.rules.encryption.localkms.local_driver import LocalKmsDriver

# The local driver HKDF-derives an AES key from this plain UTF-8 passphrase
# (it does not need to literally be 32 bytes; it is not used as a raw key).
os.environ['LOCAL_SECRET'] = "my_32_byte_local_secret_key_12345!"

FieldEncryptionExecutor.register()
LocalKmsDriver.register()
print("Successfully registered Local KMS Driver for CSFLE.")


%6|1786457947.587|GETSUBSCRIPTIONS|rdkafka#producer-3| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to vqkCYrgaSqGHaLaqdYnP0A


Created topic: bnpl_transactions_c52661_secure
Registered CSFLE Schema ID: 100015
Successfully registered Local KMS Driver for CSFLE.


### Produce a record with an encrypted field

`national_id` is tagged `PII` in the schema, so the `ENCRYPT` rule transforms it into ciphertext client-side before the record ever leaves the process — the plaintext never touches the network or the broker.

**Example:** sending `{"customer_id": "CUST-999", "national_id": "123-456-7890"}` results in `national_id` being stored as an encrypted blob; `customer_id` stays plaintext since it isn't tagged.

In [11]:
# Produce Encrypted Data
json_serializer_csfle = JSONSerializer(
    schema_str=schema_csfle_str, 
    schema_registry_client=sr_client,
    conf={'auto.register.schemas': False, 'use.latest.version': True}
)

csfle_data = {
    "customer_id": "CUST-999",
    "national_id": "123-456-7890" # This will be encrypted locally before sending!
}

try:
    ctx = SerializationContext(topic_csfle, MessageField.VALUE)
    serialized_csfle = json_serializer_csfle(csfle_data, ctx)
    producer.produce(topic=topic_csfle, value=serialized_csfle, on_delivery=delivery_report)
    producer.flush()
    print("Successfully produced encrypted CSFLE data to Confluent Cloud.")
except Exception as e:
    print(f"Encryption Produce Error: {e}")

Message delivered to bnpl_transactions_c52661_secure [0] @ offset 0
Successfully produced encrypted CSFLE data to Confluent Cloud.


## 8. Consumer Without CSFLE (Raw byte view)
We will attempt to consume the topic without rules/decryption capabilities. The `national_id` is protected at rest and over the wire.

In [12]:
consumer_conf_no_csfle = kafka_conf.copy()
consumer_conf_no_csfle['group.id'] = f"workshop_no_csfle_{uuid.uuid4()}"
consumer_conf_no_csfle['auto.offset.reset'] = 'earliest'

raw_consumer = Consumer(consumer_conf_no_csfle)
raw_consumer.subscribe([topic_csfle])

print("Consuming messages WITHOUT CSFLE decryption...")
msg = raw_consumer.poll(10.0)

if msg is None:
    print("No message received.")
elif msg.error():
    print(f"Consumer error: {msg.error()}")
else:
    raw_bytes = msg.value()
    print(f"\nRaw Byte Array (Encrypted Data on Wire): {raw_bytes[:80]}...")
    print("-> Note: You cannot read '123-456-7890' natively from this byte stream!")

raw_consumer.close()

Consuming messages WITHOUT CSFLE decryption...


%6|1786457950.922|GETSUBSCRIPTIONS|rdkafka#consumer-4| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to 3apT8DwyScSv4RgJ/arzDg



Raw Byte Array (Encrypted Data on Wire): b'\x00\x00\x01\x86\xaf{"customer_id":"CUST-999","national_id":"7PsXTtCzgBSWwSZY8avyZwKyKZ68opx/CZ'...
-> Note: You cannot read '123-456-7890' natively from this byte stream!


## 9. Consumer to Read Encrypted Data (With CSFLE)
Now we use a correctly configured `JSONDeserializer` which utilizes the KMS decryption rules attached to the schema to read the plaintext.

In [13]:
from confluent_kafka.schema_registry.json_schema import JSONDeserializer

json_deserializer_csfle = JSONDeserializer(
    schema_str=schema_csfle_str,
    schema_registry_client=sr_client,
    conf={'use.latest.version': True} # Automatically fetches the rule structure
)

consumer_conf_csfle = kafka_conf.copy()
consumer_conf_csfle['group.id'] = f"workshop_csfle_secure_{uuid.uuid4()}"
consumer_conf_csfle['auto.offset.reset'] = 'earliest'

secure_consumer = Consumer(consumer_conf_csfle)
secure_consumer.subscribe([topic_csfle])

print("Consuming messages WITH CSFLE decryption rules...")
for _ in range(5):
    msg = secure_consumer.poll(3.0)
    if msg is None:
        continue
    if msg.error():
        print(f"Consumer error: {msg.error()}")
        break
    
    ctx = SerializationContext(msg.topic(), MessageField.VALUE)
    try:
        # Deserializer decrypts automatically under the hood via Rule Registry
        decrypted_data = json_deserializer_csfle(msg.value(), ctx)
        print(f"\nSuccessfully decrypted plaintext data: {decrypted_data}")
        break
    except Exception as e:
        print(f"Deserialization/Decryption Error: {e}")
        
secure_consumer.close()

Consuming messages WITH CSFLE decryption rules...


%6|1786457954.593|GETSUBSCRIPTIONS|rdkafka#consumer-5| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to jLex17/DSvK1AuFOGaNjQQ



Successfully decrypted plaintext data: {'customer_id': 'CUST-999', 'national_id': '123-456-7890'}


%5|1786460744.602|REQTMOUT|rdkafka#producer-2| [thrd:sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:90]: sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:9092/0: Timed out MetadataRequest in flight (after 1051759ms, timeout #0)
%5|1786460744.602|REQTMOUT|rdkafka#producer-2| [thrd:sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:90]: sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:9092/0: Timed out PushTelemetryRequest in flight (after 1051759ms, timeout #1)
%5|1786460744.602|REQTMOUT|rdkafka#producer-2| [thrd:sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:90]: sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:9092/0: Timed out MetadataRequest in flight (after 1050756ms, timeout #2)
%5|1786460744.602|REQTMOUT|rdkafka#producer-2| [thrd:sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:90]: sasl_ssl://b0-pkc-n5wwrz.asia-southeast2.gcp.confluent.cloud:9092/0: Timed out MetadataRequest in flight (after 10

## Cleanup

Set `CLEANUP = True` and re-run this cell to tear down everything this notebook created:
the two topics (`bnpl_transactions_<you>` and its `_secure` CSFLE topic) and their
schema subjects. Safe to re-run -- deleting an already-deleted topic/subject is reported,
not treated as fatal.

In [ ]:
CLEANUP = False

if not CLEANUP:
    print(f"ℹ️ Cleanup disabled. Set CLEANUP = True and re-run to tear down:")
    print(f"     topics   : {topic_name}, {topic_csfle}")
    print(f"     subjects : {subject_name}, {csfle_subject_name}")
else:
    admin = AdminClient(kafka_conf)

    for subj in (subject_name, csfle_subject_name):
        for permanent in (False, True):
            try:
                sr_client.delete_subject(subj, permanent=permanent)
                print(f"✅ subject {subj} ({'hard' if permanent else 'soft'} delete)")
            except Exception as e:
                print(f"ℹ️ subject {subj} ({'hard' if permanent else 'soft'}): {str(e)[:90]}")

    for name, fut in admin.delete_topics([topic_name, topic_csfle]).items():
        try:
            fut.result(timeout=60)
            print(f"✅ topic {name} deleted")
        except Exception as e:
            print(f"ℹ️ topic {name}: {str(e)[:90]}")

    print(f"\n✅ Cleanup complete.")